# XGBoost on PySpark - 完整预处理流水线（类别/数值特征分离）

## 核心设计
1. **配置文件** (`config.yaml`): 定义类别特征、数值特征列表
2. **训练/验证拆分**: 数据分为训练集 (80%) 和验证集 (20%)
3. **训练集预处理**: 类别特征 (StringIndexer → OneHotEncoder) + 数值特征 (Imputer → StandardScaler) → 组装为特征向量
4. **预处理要素保存到 S3**: PipelineModel 保存至 `s3a://preprocessing/fraud_pipeline`
5. **验证集预处理**: 从 S3 加载 PipelineModel，对验证数据 transform（使用训练集的编码/标准化参数）
6. **XGBoost 训练**: 在预处理后的训练数据上分布式训练
7. **评估**: 在预处理后的验证数据上分布式评估

**全程基于 Spark DataFrame 分布式处理，不使用 `toPandas()`。**

In [1]:
import sys, os, glob

# apache/spark 镜像中 pyspark 不在 pip 路径，需要手动添加
sys.path.insert(0, "/opt/spark/python")
py4j_zip = glob.glob("/opt/spark/python/lib/py4j-*.zip")
if py4j_zip:
    sys.path.insert(0, py4j_zip[0])

# 确保 executor 上也安装了 pyarrow（XGBoost mapInPandas 需要）
# 若 executor 报 "PyArrow >= 4.0.0 must be installed"，请执行：
# docker exec spark-worker-1 pip install -i https://pypi.tuna.tsinghua.edu.cn/simple pyarrow
# docker exec spark-worker-2 pip install -i https://pypi.tuna.tsinghua.edu.cn/simple pyarrow

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("XGBoost-Preprocessing-Pipeline") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "2g") \
    .config("spark.executor.cores", "1") \
    .config("spark.driver.memory", "2g") \
    .config("spark.task.cpus", "1") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.fast.upload", "true") \
    .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/21 09:57:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


----------------------------------------
Exception happened during processing of request from ('127.0.0.1', 35102)
Traceback (most recent call last):
  File "/usr/lib/python3.8/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.8/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.8/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.8/socketserver.py", line 747, in __init__
    self.handle()
  File "/opt/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/opt/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
  File "/opt/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates = read_int(self.rfile)
  File "/opt/spark/python/pyspark/serializers.py", line 59

In [2]:
# 读取配置文件
import yaml

config_path = "/home/feynman/work/config.yaml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

cat_features = config["features"]["categorical"]
num_features = config["features"]["numerical"]
label_col = config["features"]["label_col"]
drop_cols = config["features"]["drop_cols"]

print("=" * 60)
print("配置文件加载成功！")
print("=" * 60)
print(f"类别特征 ({len(cat_features)} 个): {cat_features}")
print(f"数值特征 ({len(num_features)} 个): {num_features}")
print(f"目标列: {label_col}")
print(f"丢弃列: {drop_cols}")

# 将配置也保存到 S3，方便后续审计
import boto3
s3_client = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin",
    region_name="us-east-1",
)
# 确保 bucket 存在
for bucket in ["datasets", "models", "preprocessing"]:
    try:
        s3_client.create_bucket(Bucket=bucket)
    except Exception:
        pass

s3_client.put_object(
    Bucket="preprocessing",
    Key="config.yaml",
    Body=yaml.dump(config),
)
print("\n配置已备份到 S3: s3a://preprocessing/config.yaml")

配置文件加载成功！
类别特征 (5 个): ['payment_method', 'merchant_category', 'transaction_hour_bucket', 'device_type', 'card_present']
数值特征 (20 个): ['amount', 'amount_ratio_to_avg', 'transaction_count_1h', 'transaction_count_24h', 'transaction_count_7d', 'user_age', 'account_age_days', 'credit_score', 'login_failures_24h', 'device_change_count_7d', 'merchant_risk_score', 'merchant_avg_amount', 'merchant_distance_km', 'hour', 'day_of_week', 'is_weekend', 'amount_zscore', 'amount_log', 'velocity_amount', 'amount_per_transaction']
目标列: is_fraud
丢弃列: ['transaction_id', 'timestamp']

配置已备份到 S3: s3a://preprocessing/config.yaml


In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType, IntegerType

n_samples = 100000

print(f"使用 Spark 分布式生成 {n_samples} 条样本...")

# ============================================================
# 策略：使用 spark.range 生成 ID，然后用 rand() 和 when/otherwise
#       在分布式环境下生成所有特征。这样避免 pickle 序列化问题。
# ============================================================

df = spark.range(0, n_samples).repartition(4)

# 基础随机数种子列
df = df.withColumn("r0", F.rand(42))
df = df.withColumn("r1", F.rand(43))
df = df.withColumn("r2", F.rand(44))
df = df.withColumn("r3", F.rand(45))
df = df.withColumn("r4", F.rand(46))
df = df.withColumn("r5", F.rand(47))

# ============================================================
# 类别特征
# ============================================================
# payment_method: credit_card(0.35), debit_card(0.25), bank_transfer(0.15), wallet(0.20), cash(0.05)
df = df.withColumn("payment_method",
    F.when(F.col("r0") < 0.35, F.lit("credit_card"))
     .when(F.col("r0") < 0.60, F.lit("debit_card"))
     .when(F.col("r0") < 0.75, F.lit("bank_transfer"))
     .when(F.col("r0") < 0.95, F.lit("wallet"))
     .otherwise(F.lit("cash"))
)

# merchant_category: 8 categories
df = df.withColumn("merchant_category",
    F.when(F.col("r1") < 0.25, F.lit("retail"))
     .when(F.col("r1") < 0.45, F.lit("food"))
     .when(F.col("r1") < 0.55, F.lit("travel"))
     .when(F.col("r1") < 0.65, F.lit("entertainment"))
     .when(F.col("r1") < 0.73, F.lit("healthcare"))
     .when(F.col("r1") < 0.83, F.lit("utility"))
     .when(F.col("r1") < 0.90, F.lit("education"))
     .otherwise(F.lit("other"))
)

# transaction_hour_bucket: night/morning/afternoon/evening
df = df.withColumn("transaction_hour_bucket",
    F.when(F.col("r2") < 0.15, F.lit("night"))
     .when(F.col("r2") < 0.45, F.lit("morning"))
     .when(F.col("r2") < 0.75, F.lit("afternoon"))
     .otherwise(F.lit("evening"))
)

# device_type: mobile/desktop/tablet/other
df = df.withColumn("device_type",
    F.when(F.col("r3") < 0.45, F.lit("mobile"))
     .when(F.col("r3") < 0.80, F.lit("desktop"))
     .when(F.col("r3") < 0.92, F.lit("tablet"))
     .otherwise(F.lit("other"))
)

# card_present: yes/no
df = df.withColumn("card_present",
    F.when(F.col("r4") < 0.70, F.lit("yes")).otherwise(F.lit("no"))
)

# ============================================================
# 数值特征（使用 randn 近似 lognormal/normal）
# ============================================================
# 使用 Box-Muller 近似: randn ≈ sqrt(-2*log(rand))*cos(2*pi*rand)
# 简化：使用多个 rand() 的和来近似正态分布（中心极限定理）

df = df.withColumn("rn6", F.rand(48))
df = df.withColumn("rn7", F.rand(49))
df = df.withColumn("rn8", F.rand(50))
df = df.withColumn("rn9", F.rand(51))

# 近似标准正态分布：sum of 6 uniforms - 3
randn_approx = (F.col("r0") + F.col("r1") + F.col("r2") + 
                F.col("r3") + F.col("r4") + F.col("r5") - 3.0) * 1.8

# amount: lognormal(4.0, 0.8) ≈ exp(4.0 + 0.8 * randn)
df = df.withColumn("amount", F.exp(4.0 + 0.8 * randn_approx))
# amount_ratio_to_avg: lognormal(0.0, 0.5) ≈ exp(0.5 * randn)
df = df.withColumn("amount_ratio_to_avg", F.exp(0.5 * (F.col("rn6") + F.col("rn7") - 1.0) * 1.8))
# transaction counts: Poisson approximated by rand
df = df.withColumn("transaction_count_1h", F.floor(F.col("r0") * 5))
df = df.withColumn("transaction_count_24h", F.floor(F.col("r1") * 30))
df = df.withColumn("transaction_count_7d", F.floor(F.col("r2") * 150))
# user_age: 18-80
df = df.withColumn("user_age", F.floor(F.col("r3") * 62 + 18))
# account_age_days: 1-3650
df = df.withColumn("account_age_days", F.floor(F.col("r4") * 3649 + 1))
# credit_score: 300-850, mean ~680
df = df.withColumn("credit_score", 
    F.least(F.lit(850.0), F.greatest(F.lit(300.0), 680.0 + 80.0 * randn_approx)))
# login_failures_24h: Poisson(0.3) ≈ rand < prob
df = df.withColumn("login_failures_24h", 
    F.when(F.col("rn6") < 0.74, F.lit(0.0))
     .when(F.col("rn6") < 0.96, F.lit(1.0))
     .otherwise(F.lit(2.0)))
# device_change_count_7d
df = df.withColumn("device_change_count_7d",
    F.when(F.col("rn7") < 0.61, F.lit(0.0))
     .when(F.col("rn7") < 0.91, F.lit(1.0))
     .otherwise(F.lit(2.0)))
# merchant_risk_score: 0-1, mean ~0.3
df = df.withColumn("merchant_risk_score",
    F.least(F.lit(1.0), F.greatest(F.lit(0.0), 0.3 + 0.2 * randn_approx)))
# merchant_avg_amount
df = df.withColumn("merchant_avg_amount", F.exp(3.5 + 0.6 * randn_approx))
# merchant_distance_km
df = df.withColumn("merchant_distance_km", F.exp(2.0 + 1.0 * randn_approx))
# hour: 0-23
df = df.withColumn("hour", F.floor(F.col("rn8") * 24))
# day_of_week: 0-6
df = df.withColumn("day_of_week", F.floor(F.col("rn9") * 7))
# is_weekend
df = df.withColumn("is_weekend",
    F.when(F.col("day_of_week") >= 5, F.lit(1.0)).otherwise(F.lit(0.0)))
# amount_zscore
df = df.withColumn("amount_zscore", randn_approx)
# amount_log
df = df.withColumn("amount_log", F.log1p(F.col("amount")))
# velocity_amount
df = df.withColumn("velocity_amount", F.col("transaction_count_1h") * F.col("amount") / 100.0)
# amount_per_transaction
df = df.withColumn("amount_per_transaction",
    F.col("transaction_count_7d") * F.col("merchant_avg_amount") / (F.col("transaction_count_7d") + 1.0))

# ============================================================
# 生成标签
# ============================================================
logit_expr = (
    -3.0
    + 1.5 * F.when(F.col("payment_method") == "wallet", F.lit(1.0)).otherwise(F.lit(0.0))
    + 1.2 * F.when(F.col("payment_method") == "cash", F.lit(1.0)).otherwise(F.lit(0.0))
    + 0.8 * F.when(F.col("merchant_category") == "travel", F.lit(1.0)).otherwise(F.lit(0.0))
    + 0.6 * F.when(F.col("merchant_category") == "entertainment", F.lit(1.0)).otherwise(F.lit(0.0))
    + 1.0 * F.when(F.col("card_present") == "no", F.lit(1.0)).otherwise(F.lit(0.0))
    + 2.0 * F.when(F.col("amount_zscore") > 2.5, F.lit(1.0)).otherwise(F.lit(0.0))
    + 1.5 * F.when(F.col("amount_ratio_to_avg") > 5.0, F.lit(1.0)).otherwise(F.lit(0.0))
    + 0.8 * F.when(F.col("transaction_count_1h") > 3, F.lit(1.0)).otherwise(F.lit(0.0))
    + 0.5 * F.when(F.col("merchant_risk_score") > 0.7, F.lit(1.0)).otherwise(F.lit(0.0))
    - 0.5 * F.when(F.col("card_present") == "yes", F.lit(1.0)).otherwise(F.lit(0.0))
    - 0.3 * F.when(F.col("payment_method") == "credit_card", F.lit(1.0)).otherwise(F.lit(0.0))
)

prob = 1.0 / (1.0 + F.exp(-logit_expr))

# 基于概率生成标签（添加少量噪声）
df = df.withColumn("rand_label", F.rand(99))
df = df.withColumn(label_col,
    F.when(F.col("rand_label") < prob * 0.992, F.lit(1))
     .when(F.col("rand_label") > prob + (1 - prob) * 0.008, F.lit(1))
     .otherwise(F.lit(0))
)

# ============================================================
# 添加 ID 列和时间戳列（之后会被丢弃）
# ============================================================
df = df.withColumn("transaction_id", F.concat(F.lit("TXN_"), F.lpad(F.col("id").cast("string"), 8, "0")))
df = df.withColumn("timestamp",
    F.concat(
        F.lit("2024-01-"),
        F.lpad(F.floor(F.col("rn8") * 28 + 1).cast("string"), 2, "0"),
        F.lit(" "),
        F.lpad(F.floor(F.col("rn9") * 24).cast("string"), 2, "0"),
        F.lit(":"),
        F.lpad(F.floor(F.col("r0") * 60).cast("string"), 2, "0"),
        F.lit(":"),
        F.lpad(F.floor(F.col("r1") * 60).cast("string"), 2, "0")
    )
)

# ============================================================
# 选择最终列（按 schema 顺序）
# ============================================================
final_cols = [
    "transaction_id", "timestamp",
] + cat_features + num_features + [label_col]

df = df.select(final_cols)

# 统计信息
total_count = df.count()
fraud_count = df.filter(F.col(label_col) == 1).count()
print(f"总样本: {total_count}")
print(f"欺诈样本: {fraud_count} ({fraud_count/total_count*100:.2f}%)")
print(f"正常样本: {total_count - fraud_count} ({(1-fraud_count/total_count)*100:.2f}%)")
print(f"DataFrame 分区数: {df.rdd.getNumPartitions()}")

# 写入 S3 为 Parquet
s3_raw_path = "s3a://datasets/fraud_detection_raw.parquet"
df.write.mode("overwrite").parquet(s3_raw_path)
print(f"原始数据已写入 S3: {s3_raw_path}")

# 释放内存
df.unpersist() if hasattr(df, "unpersist") else None
print("本地 DataFrame 已释放，后续从 S3 读取")

使用 Spark 分布式生成 100000 条样本...


总样本: 100000
欺诈样本: 99191 (99.19%)
正常样本: 809 (0.81%)
DataFrame 分区数: 4


26/06/21 09:57:28 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/06/21 09:57:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

原始数据已写入 S3: s3a://datasets/fraud_detection_raw.parquet
本地 DataFrame 已释放，后续从 S3 读取


In [4]:
# ============================================================
# 阶段 2: 从 S3 读取数据，拆分为训练集和验证集
# ============================================================
print("=" * 60)
print("从 S3 读取数据并拆分训练/验证集")
print("=" * 60)

df = spark.read.parquet("s3a://datasets/fraud_detection_raw.parquet")
print(f"从 S3 读取: {df.count()} 行, {df.rdd.getNumPartitions()} 分区")

# 分布式拆分为 训练集 (80%) 和 验证集 (20%)
train_df, val_df = df.randomSplit([0.8, 0.2], seed=42)

# 重分区并缓存
train_df = train_df.repartition(4).cache()
val_df = val_df.repartition(4).cache()

# 触发缓存
train_count = train_df.count()
val_count = val_df.count()

# 分布式统计类别分布
print(f"\n训练集: {train_count} 行")
train_df.groupBy(label_col).count().orderBy(label_col).show()

print(f"验证集: {val_count} 行")
val_df.groupBy(label_col).count().orderBy(label_col).show()

# 验证分区分布
train_partition_sizes = train_df.rdd.glom().map(len).collect()
val_partition_sizes = val_df.rdd.glom().map(len).collect()
print(f"训练集分区行数: {train_partition_sizes}")
print(f"验证集分区行数: {val_partition_sizes}")

# 展示几条样例
print("\n训练集样例:")
train_df.select(cat_features[:3] + num_features[:3] + [label_col]).show(5, truncate=False)

从 S3 读取数据并拆分训练/验证集
从 S3 读取: 100000 行, 2 分区



训练集: 80060 行
+--------+-----+
|is_fraud|count|
+--------+-----+
|       0|  640|
|       1|79420|
+--------+-----+

验证集: 19940 行
+--------+-----+
|is_fraud|count|
+--------+-----+
|       0|  169|
|       1|19771|
+--------+-----+



训练集分区行数: [20015, 20015, 20015, 20015]
验证集分区行数: [4985, 4985, 4985, 4985]

训练集样例:
+--------------+-----------------+-----------------------+------------------+-------------------+--------------------+--------+
|payment_method|merchant_category|transaction_hour_bucket|amount            |amount_ratio_to_avg|transaction_count_1h|is_fraud|
+--------------+-----------------+-----------------------+------------------+-------------------+--------------------+--------+
|cash          |retail           |afternoon              |24.06292914148655 |1.0216180767911363 |4                   |1       |
|credit_card   |other            |afternoon              |49.61455155959176 |0.72180362933317   |0                   |1       |
|bank_transfer |food             |evening                |57.211910388234266|1.5982178700727399 |3                   |1       |
|credit_card   |retail           |night                  |12.386039834148471|1.9784974347463427 |0                   |1       |
|wallet        |food    

In [5]:
import time
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder,
    Imputer, StandardScaler, VectorAssembler
)

print("=" * 60)
print("阶段 3: 对训练数据进行预处理")
print("类别特征: StringIndexer → OneHotEncoder")
print("数值特征: VectorAssembler → Imputer(median) → StandardScaler")
print("=" * 60)

# ============================================================
# 类别特征预处理: StringIndexer → OneHotEncoder
# ============================================================
cat_indexers = []
cat_encoders = []
cat_encoded_cols = []

print(f"\n类别特征 ({len(cat_features)} 个):")
for col in cat_features:
    indexed_col = f"{col}_idx"
    encoded_col = f"{col}_ohe"

    indexer = StringIndexer(
        inputCol=col,
        outputCol=indexed_col,
        handleInvalid="keep",
    )
    encoder = OneHotEncoder(
        inputCol=indexed_col,
        outputCol=encoded_col,
        dropLast=False,
    )

    cat_indexers.append(indexer)
    cat_encoders.append(encoder)
    cat_encoded_cols.append(encoded_col)
    print(f"  {col} → {indexed_col} → {encoded_col}")

# ============================================================
# 数值特征预处理: 每列单独 Imputer → VectorAssembler → StandardScaler
# 注意：Imputer 用于填充缺失值（每列单独处理），
#      VectorAssembler 合并为向量，StandardScaler 标准化向量
# ============================================================
print(f"\n数值特征 ({len(num_features)} 个):")

# 对每个数值列单独做 Imputer（填充缺失值）
num_imputers = []
num_imputed_cols = []
for col in num_features:
    imputed_col = f"{col}_imputed"
    imputer = Imputer(
        inputCols=[col],
        outputCols=[imputed_col],
        strategy="median",
    )
    num_imputers.append(imputer)
    num_imputed_cols.append(imputed_col)

print(f"  {len(num_features)} 个 Imputer (每列单独填充缺失值, strategy=median)")

# 将填充后的数值列合并为向量
num_assembler = VectorAssembler(
    inputCols=num_imputed_cols,
    outputCol="num_features_raw",
    handleInvalid="skip",
)

print(f"  {len(num_imputed_cols)} 列 → num_features_raw (VectorAssembler)")

# 标准化数值向量
num_scaler = StandardScaler(
    inputCol="num_features_raw",
    outputCol="num_features_scaled",
    withStd=True,
    withMean=True,
)

print(f"  num_features_raw → num_features_scaled (StandardScaler, withMean+withStd)")

# ============================================================
# 最终组装：OHE编码的类别特征 + 标准化的数值特征
# ============================================================
final_assembler = VectorAssembler(
    inputCols=cat_encoded_cols + ["num_features_scaled"],
    outputCol="features",
    handleInvalid="skip",
)

# ============================================================
# 构建 Pipeline
# 顺序: StringIndexer → OneHotEncoder → 数值Imputer → VectorAssembler → StandardScaler → 最终VectorAssembler
# ============================================================
stages = cat_indexers + cat_encoders + num_imputers + [
    num_assembler,
    num_scaler,
    final_assembler,
]

pipeline = Pipeline(stages=stages)

print(f"\nPipeline 共 {len(stages)} 个阶段:")
for i, stage in enumerate(stages):
    print(f"  [{i+1}] {type(stage).__name__}")

# ============================================================
# 在训练数据上 fit Pipeline
# ============================================================
print(f"\n在训练集上拟合 Pipeline ({train_count} 行)...")
start_time = time.time()
pipeline_model = pipeline.fit(train_df)
fit_time = time.time() - start_time
print(f"Pipeline 拟合完成！耗时: {fit_time:.2f} 秒")

# Transform 训练数据
print(f"对训练集进行 transform...")
start_time = time.time()
train_preprocessed = pipeline_model.transform(train_df).select("features", label_col)
train_preprocessed = train_preprocessed.repartition(4).cache()
train_processed_count = train_preprocessed.count()
transform_time = time.time() - start_time
print(f"训练集 transform 完成！耗时: {transform_time:.2f} 秒")
print(f"预处理后训练集: {train_processed_count} 行")

# 查看特征向量
print("\n预处理后的特征向量样例 (前3行，显示前80个元素):")
train_preprocessed.show(3, truncate=80)

# 特征维度
train_feat_size = len(train_preprocessed.select("features").first()["features"])
print(f"特征向量维度: {train_feat_size}")

阶段 3: 对训练数据进行预处理
类别特征: StringIndexer → OneHotEncoder
数值特征: VectorAssembler → Imputer(median) → StandardScaler

类别特征 (5 个):
  payment_method → payment_method_idx → payment_method_ohe
  merchant_category → merchant_category_idx → merchant_category_ohe
  transaction_hour_bucket → transaction_hour_bucket_idx → transaction_hour_bucket_ohe
  device_type → device_type_idx → device_type_ohe
  card_present → card_present_idx → card_present_ohe

数值特征 (20 个):
  20 个 Imputer (每列单独填充缺失值, strategy=median)
  20 列 → num_features_raw (VectorAssembler)
  num_features_raw → num_features_scaled (StandardScaler, withMean+withStd)

Pipeline 共 33 个阶段:
  [1] StringIndexer
  [2] StringIndexer
  [3] StringIndexer
  [4] StringIndexer
  [5] StringIndexer
  [6] OneHotEncoder
  [7] OneHotEncoder
  [8] OneHotEncoder
  [9] OneHotEncoder
  [10] OneHotEncoder
  [11] Imputer
  [12] Imputer
  [13] Imputer
  [14] Imputer
  [15] Imputer
  [16] Imputer
  [17] Imputer
  [18] Imputer
  [19] Imputer
  [20] Imputer
  [21] Imput

Pipeline 拟合完成！耗时: 7.57 秒
对训练集进行 transform...


[Stage 163:============================>                            (2 + 2) / 4]

训练集 transform 完成！耗时: 1.74 秒
预处理后训练集: 80060 行

预处理后的特征向量样例 (前3行，显示前80个元素):
+--------------------------------------------------------------------------------+--------+
|                                                                        features|is_fraud|
+--------------------------------------------------------------------------------+--------+
|(48,[2,11,15,20,25,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,4...|       1|
|(48,[4,6,16,20,26,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47...|       1|
|(48,[0,8,18,20,25,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47...|       1|
+--------------------------------------------------------------------------------+--------+
only showing top 3 rows

特征向量维度: 48


In [6]:
# ============================================================
# 阶段 4: 将预处理要素保存到 S3
# ============================================================
import json

print("=" * 60)
print("保存预处理 Pipeline 和元数据到 S3")
print("=" * 60)

pipeline_s3_path = "s3a://preprocessing/fraud_pipeline"
print(f"保存 PipelineModel 到: {pipeline_s3_path}")

start_time = time.time()
pipeline_model.write().overwrite().save(pipeline_s3_path)
save_time = time.time() - start_time
print(f"保存完成！耗时: {save_time:.2f} 秒")

# 保存预处理元数据
metadata = {
    "pipeline_stages": [str(type(s).__name__) for s in stages],
    "categorical_features": cat_features,
    "numerical_features": num_features,
    "categorical_encoded_cols": cat_encoded_cols,
    "label_col": label_col,
    "num_stages": len(stages),
    "feature_dim": train_feat_size,
}

s3_client.put_object(
    Bucket="preprocessing",
    Key="pipeline_metadata.json",
    Body=json.dumps(metadata, indent=2),
)
print("元数据已保存到: s3a://preprocessing/pipeline_metadata.json")

# 验证 S3 上的文件
print("\nS3 preprocessing bucket 内容:")
response = s3_client.list_objects_v2(Bucket="preprocessing")
if "Contents" in response:
    for obj in response["Contents"]:
        print(f"  s3a://preprocessing/{obj['Key']} ({obj['Size']} bytes)")

# 提取每个类别特征的索引映射
print("\n类别特征 StringIndexer 映射（用于推理时解码）:")
for i, col in enumerate(cat_features):
    indexer_model = pipeline_model.stages[i]
    labels = indexer_model.labels
    print(f"  {col}: {list(labels)}")

保存预处理 Pipeline 和元数据到 S3
保存 PipelineModel 到: s3a://preprocessing/fraud_pipeline
保存完成！耗时: 12.35 秒
元数据已保存到: s3a://preprocessing/pipeline_metadata.json

S3 preprocessing bucket 内容:
  s3a://preprocessing/config.yaml (613 bytes)
  s3a://preprocessing/fraud_pipeline/metadata/_SUCCESS (0 bytes)
  s3a://preprocessing/fraud_pipeline/metadata/part-00000 (1020 bytes)
  s3a://preprocessing/fraud_pipeline/stages/00_StringIndexer_07ef318282e6/data/_SUCCESS (0 bytes)
  s3a://preprocessing/fraud_pipeline/stages/00_StringIndexer_07ef318282e6/data/part-00000-fcb737ae-de51-4761-9967-efceeb57a6ea-c000.snappy.parquet (757 bytes)
  s3a://preprocessing/fraud_pipeline/stages/00_StringIndexer_07ef318282e6/metadata/_SUCCESS (0 bytes)
  s3a://preprocessing/fraud_pipeline/stages/00_StringIndexer_07ef318282e6/metadata/part-00000 (367 bytes)
  s3a://preprocessing/fraud_pipeline/stages/01_StringIndexer_30f91830a106/data/_SUCCESS (0 bytes)
  s3a://preprocessing/fraud_pipeline/stages/01_StringIndexer_30f91830a106/data/

In [7]:
# ============================================================
# 阶段 5: 从 S3 加载预处理 Pipeline，对验证集进行预处理
#         （使用训练集的编码/标准化参数）
# ============================================================
from pyspark.ml import PipelineModel

print("=" * 60)
print("从 S3 加载预处理 Pipeline，对验证集进行 transform")
print("=" * 60)

pipeline_s3_path = "s3a://preprocessing/fraud_pipeline"
loaded_pipeline = PipelineModel.load(pipeline_s3_path)
print(f"PipelineModel 加载成功！阶段数: {len(loaded_pipeline.stages)}")

# 验证阶段类型一致性
for i, stage in enumerate(loaded_pipeline.stages):
    print(f"  [{i}] {type(stage).__name__}")

# 对验证集进行 transform（使用训练集的参数，不做任何 fit）
print(f"\n对验证集进行 transform ({val_count} 行)...")
start_time = time.time()
val_preprocessed = loaded_pipeline.transform(val_df).select("features", label_col)
val_preprocessed = val_preprocessed.repartition(4).cache()
val_processed_count = val_preprocessed.count()
transform_time = time.time() - start_time
print(f"验证集 transform 完成！耗时: {transform_time:.2f} 秒")
print(f"预处理后验证集: {val_processed_count} 行")

# 验证预处理后的特征一致性
print("\n验证集预处理后的特征向量样例:")
val_preprocessed.show(3, truncate=80)

# 检查特征维度一致性
val_feat_size = len(val_preprocessed.select("features").first()["features"])
print(f"\n特征维度检查:")
print(f"  训练集特征维度: {train_feat_size}")
print(f"  验证集特征维度: {val_feat_size}")
print(f"  维度一致: {'YES ✓' if train_feat_size == val_feat_size else 'NO ✗'}")

从 S3 加载预处理 Pipeline，对验证集进行 transform
PipelineModel 加载成功！阶段数: 33
  [0] StringIndexerModel
  [1] StringIndexerModel
  [2] StringIndexerModel
  [3] StringIndexerModel
  [4] StringIndexerModel
  [5] OneHotEncoderModel
  [6] OneHotEncoderModel
  [7] OneHotEncoderModel
  [8] OneHotEncoderModel
  [9] OneHotEncoderModel
  [10] ImputerModel
  [11] ImputerModel
  [12] ImputerModel
  [13] ImputerModel
  [14] ImputerModel
  [15] ImputerModel
  [16] ImputerModel
  [17] ImputerModel
  [18] ImputerModel
  [19] ImputerModel
  [20] ImputerModel
  [21] ImputerModel
  [22] ImputerModel
  [23] ImputerModel
  [24] ImputerModel
  [25] ImputerModel
  [26] ImputerModel
  [27] ImputerModel
  [28] ImputerModel
  [29] ImputerModel
  [30] VectorAssembler
  [31] StandardScalerModel
  [32] VectorAssembler

对验证集进行 transform (19940 行)...
验证集 transform 完成！耗时: 2.07 秒
预处理后验证集: 19940 行

验证集预处理后的特征向量样例:
+--------------------------------------------------------------------------------+--------+
|                           

In [8]:
from xgboost.spark import SparkXGBClassifier

print("=" * 60)
print("阶段 6: XGBoost 分布式训练")
print("=" * 60)

# 计算正负样本比例用于 scale_pos_weight
train_pos = train_preprocessed.filter(F.col(label_col) == 1).count()
train_neg = train_preprocessed.filter(F.col(label_col) == 0).count()
scale_weight = train_neg / train_pos if train_pos > 0 else 1.0
print(f"训练集正负样本比: {train_neg}:{train_pos} = {scale_weight:.2f}:1")

xgb_classifier = SparkXGBClassifier(
    features_col="features",
    label_col=label_col,
    num_workers=2,
    n_estimators=50,
    max_depth=6,
    learning_rate=0.1,
    eval_metric="auc",
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_weight,
    missing=0.0,
    verbosity=1,
)

print("开始分布式训练...")
start_time = time.time()
xgb_model = xgb_classifier.fit(train_preprocessed)
train_time_xgb = time.time() - start_time
print(f"训练完成！耗时: {train_time_xgb:.2f} 秒")

/usr/local/lib/python3.8/dist-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


阶段 6: XGBoost 分布式训练
训练集正负样本比: 640:79420 = 0.01:1
开始分布式训练...


2026-06-21 09:58:13,179 INFO XGBoost-PySpark: _fit Running xgboost-2.1.4 on 2 workers with
	booster params: {'objective': 'binary:logistic', 'colsample_bytree': 0.8, 'device': 'cpu', 'eval_metric': 'auc', 'learning_rate': 0.1, 'max_depth': 6, 'reg_alpha': 0.1, 'reg_lambda': 1.0, 'scale_pos_weight': 0.008058423570888944, 'subsample': 0.8, 'verbosity': 1, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 50}
	dmatrix_kwargs: {'nthread': 1, 'missing': 0.0}
2026-06-21 09:58:18,506 INFO XGBoost-PySpark: _fit Finished xgboost training!   


训练完成！耗时: 5.94 秒


In [9]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

print("=" * 60)
print("阶段 7: 分布式评估（在验证集上）")
print("=" * 60)

# 分布式预测
val_predictions = xgb_model.transform(val_preprocessed)
val_predictions.cache()
pred_count = val_predictions.count()
print(f"验证集预测样本数: {pred_count}")

# AUC (ROC)
auc_roc = BinaryClassificationEvaluator(
    labelCol=label_col, rawPredictionCol="rawPrediction", metricName="areaUnderROC"
).evaluate(val_predictions)

# AUC (PR)
auc_pr = BinaryClassificationEvaluator(
    labelCol=label_col, rawPredictionCol="rawPrediction", metricName="areaUnderPR"
).evaluate(val_predictions)

# Accuracy
accuracy = MulticlassClassificationEvaluator(
    labelCol=label_col, predictionCol="prediction", metricName="accuracy"
).evaluate(val_predictions)

# F1
f1 = MulticlassClassificationEvaluator(
    labelCol=label_col, predictionCol="prediction", metricName="f1"
).evaluate(val_predictions)

print(f"\n模型评估结果:")
print(f"  AUC (ROC):  {auc_roc:.4f}")
print(f"  AUC (PR):   {auc_pr:.4f}")
print(f"  Accuracy:   {accuracy:.4f}")
print(f"  F1 Score:   {f1:.4f}")

# 混淆矩阵（纯 Spark 分布式计算）
print(f"\n混淆矩阵:")
cm = val_predictions.groupBy(label_col, "prediction").count().orderBy(label_col, "prediction")
cm.show()

# 计算 TP/TN/FP/FN
tp = val_predictions.filter((F.col(label_col) == 1) & (F.col("prediction") == 1)).count()
tn = val_predictions.filter((F.col(label_col) == 0) & (F.col("prediction") == 0)).count()
fp = val_predictions.filter((F.col(label_col) == 0) & (F.col("prediction") == 1)).count()
fn = val_predictions.filter((F.col(label_col) == 1) & (F.col("prediction") == 0)).count()

precision_val = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall_val = tp / (tp + fn) if (tp + fn) > 0 else 0.0

print(f"\n详细指标:")
print(f"  TP={tp}, TN={tn}, FP={fp}, FN={fn}")
print(f"  Precision:    {precision_val:.4f}")
print(f"  Recall:       {recall_val:.4f}")
if (tn + fp) > 0:
    print(f"  Specificity:  {tn / (tn + fp):.4f}")

阶段 7: 分布式评估（在验证集上）
验证集预测样本数: 19940

模型评估结果:
  AUC (ROC):  0.4922
  AUC (PR):   0.9916
  Accuracy:   0.8341
  F1 Score:   0.9018

混淆矩阵:
+--------+----------+-----+
|is_fraud|prediction|count|
+--------+----------+-----+
|       0|       0.0|   25|
|       0|       1.0|  144|
|       1|       0.0| 3164|
|       1|       1.0|16607|
+--------+----------+-----+


详细指标:
  TP=16607, TN=25, FP=144, FN=3164
  Precision:    0.9914
  Recall:       0.8400
  Specificity:  0.1479


In [10]:
# 将训练好的 XGBoost 模型保存到 S3
model_s3_path = "s3a://models/xgboost_fraud_detection"
print(f"保存 XGBoost 模型到 S3: {model_s3_path}")
xgb_model.write().overwrite().save(model_s3_path)
print("模型保存成功！")

# 特征重要性
print("\n特征重要性 (Top 10, by gain):")
importance_dict = xgb_model.get_booster().get_score(importance_type="gain")
sorted_importance = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)[:10]
for feat, score in sorted_importance:
    print(f"  {feat}: {score:.4f}")

保存 XGBoost 模型到 S3: s3a://models/xgboost_fraud_detection
模型保存成功！

特征重要性 (Top 10, by gain):
  f44: 4.3714
  f18: 2.4644
  f6: 2.3265
  f39: 2.2808
  f13: 2.1417
  f11: 1.8676
  f4: 1.8251
  f12: 1.8116
  f17: 1.7800
  f9: 1.7434


In [11]:
# ============================================================
# 阶段 8: 模拟生产推理流程
# 新数据 → 从 S3 加载预处理 Pipeline → transform → 加载模型 → predict
# ============================================================
from pyspark.ml import PipelineModel
from xgboost.spark import SparkXGBClassifierModel

print("=" * 60)
print("阶段 8: 模拟生产推理")
print("从 S3 加载预处理 Pipeline + 模型，对新数据做推理")
print("=" * 60)

# 从 S3 加载预处理 Pipeline
inference_pipeline = PipelineModel.load("s3a://preprocessing/fraud_pipeline")
print("✓ 预处理 Pipeline 加载成功")

# 从 S3 加载 XGBoost 模型
inference_model = SparkXGBClassifierModel.load("s3a://models/xgboost_fraud_detection")
print("✓ XGBoost 模型加载成功")

# 生成新数据模拟生产请求（使用 Spark 分布式方式）
n_new = 5
new_df_base = spark.range(0, n_new).repartition(1)

# 手动构造测试用例
new_df_base = new_df_base.withColumn("idx", F.col("id"))

new_df_base = new_df_base.withColumn("payment_method",
    F.when(F.col("idx") == 0, F.lit("credit_card"))
     .when(F.col("idx") == 1, F.lit("wallet"))
     .when(F.col("idx") == 2, F.lit("debit_card"))
     .when(F.col("idx") == 3, F.lit("cash"))
     .otherwise(F.lit("bank_transfer"))
)

new_df_base = new_df_base.withColumn("merchant_category",
    F.when(F.col("idx") == 0, F.lit("retail"))
     .when(F.col("idx") == 1, F.lit("travel"))
     .when(F.col("idx") == 2, F.lit("food"))
     .when(F.col("idx") == 3, F.lit("entertainment"))
     .otherwise(F.lit("healthcare"))
)

new_df_base = new_df_base.withColumn("transaction_hour_bucket",
    F.when(F.col("idx") == 0, F.lit("morning"))
     .when(F.col("idx") == 1, F.lit("night"))
     .when(F.col("idx") == 2, F.lit("afternoon"))
     .when(F.col("idx") == 3, F.lit("evening"))
     .otherwise(F.lit("morning"))
)

new_df_base = new_df_base.withColumn("device_type",
    F.when(F.col("idx") < 3, F.lit("mobile")).otherwise(F.lit("desktop"))
)

new_df_base = new_df_base.withColumn("card_present",
    F.when(F.col("idx").isin([0, 2, 4]), F.lit("yes")).otherwise(F.lit("no"))
)

# 数值特征
amounts = [55.0, 3200.0, 120.0, 8000.0, 200.0]
amount_ratios = [0.8, 15.0, 1.2, 25.0, 0.9]
txn_1h = [1.0, 6.0, 0.0, 10.0, 2.0]
txn_24h = [5.0, 25.0, 8.0, 40.0, 6.0]
txn_7d = [30.0, 120.0, 40.0, 200.0, 35.0]

new_df_base = new_df_base.withColumn("amount",
    F.when(F.col("idx") == 0, F.lit(amounts[0]))
     .when(F.col("idx") == 1, F.lit(amounts[1]))
     .when(F.col("idx") == 2, F.lit(amounts[2]))
     .when(F.col("idx") == 3, F.lit(amounts[3]))
     .otherwise(F.lit(amounts[4]))
)

new_df_base = new_df_base.withColumn("amount_ratio_to_avg",
    F.when(F.col("idx") == 0, F.lit(amount_ratios[0]))
     .when(F.col("idx") == 1, F.lit(amount_ratios[1]))
     .when(F.col("idx") == 2, F.lit(amount_ratios[2]))
     .when(F.col("idx") == 3, F.lit(amount_ratios[3]))
     .otherwise(F.lit(amount_ratios[4]))
)

new_df_base = new_df_base.withColumn("transaction_count_1h",
    F.when(F.col("idx") == 0, F.lit(txn_1h[0]))
     .when(F.col("idx") == 1, F.lit(txn_1h[1]))
     .when(F.col("idx") == 2, F.lit(txn_1h[2]))
     .when(F.col("idx") == 3, F.lit(txn_1h[3]))
     .otherwise(F.lit(txn_1h[4]))
)

new_df_base = new_df_base.withColumn("transaction_count_24h",
    F.when(F.col("idx") == 0, F.lit(txn_24h[0]))
     .when(F.col("idx") == 1, F.lit(txn_24h[1]))
     .when(F.col("idx") == 2, F.lit(txn_24h[2]))
     .when(F.col("idx") == 3, F.lit(txn_24h[3]))
     .otherwise(F.lit(txn_24h[4]))
)

new_df_base = new_df_base.withColumn("transaction_count_7d",
    F.when(F.col("idx") == 0, F.lit(txn_7d[0]))
     .when(F.col("idx") == 1, F.lit(txn_7d[1]))
     .when(F.col("idx") == 2, F.lit(txn_7d[2]))
     .when(F.col("idx") == 3, F.lit(txn_7d[3]))
     .otherwise(F.lit(txn_7d[4]))
)

# 其余数值特征填充默认值
for col_name in num_features:
    if col_name not in ["amount", "amount_ratio_to_avg", "transaction_count_1h",
                         "transaction_count_24h", "transaction_count_7d"]:
        new_df_base = new_df_base.withColumn(col_name,
            F.when(F.col("idx") == 0, F.lit(35.0))  # user_age
             .when(F.col("idx") == 1, F.lit(500.0))  # account_age_days
             .otherwise(F.lit(0.0))
        )

# 添加必要列
new_df_base = new_df_base.withColumn("transaction_id",
    F.concat(F.lit("INF_"), F.lpad(F.col("id").cast("string"), 5, "0")))
new_df_base = new_df_base.withColumn("timestamp", F.lit("2024-06-21 14:30:00"))
new_df_base = new_df_base.withColumn(label_col, F.lit(0))  # placeholder

# 选择与训练数据相同的列顺序
new_df = new_df_base.select(final_cols)

# 预处理 → 预测（纯分布式）
new_preprocessed = inference_pipeline.transform(new_df)
new_predictions = inference_model.transform(new_preprocessed)

# 收集结果
results = new_predictions.select(
    "transaction_id", "payment_method", "merchant_category",
    "amount", "probability", "prediction"
).collect()

print("\n生产推理结果:")
print(f"{'ID':<10} {'支付方式':<15} {'商户类别':<15} {'金额':>10} {'欺诈概率':>10} {'预测':>8}")
print("-" * 75)
for row in results:
    fraud_prob = float(row["probability"][1])
    pred_label = "欺诈⚠" if row["prediction"] == 1 else "正常✓"
    print(f"{row['transaction_id']:<10} {row['payment_method']:<15} {row['merchant_category']:<15} {row['amount']:>10.2f} {fraud_prob:>10.4f} {pred_label:>8}")

阶段 8: 模拟生产推理
从 S3 加载预处理 Pipeline + 模型，对新数据做推理
✓ 预处理 Pipeline 加载成功
✓ XGBoost 模型加载成功

生产推理结果:
ID         支付方式            商户类别                    金额       欺诈概率       预测
---------------------------------------------------------------------------
INF_00002  debit_card      food                120.00     0.7609      欺诈⚠
INF_00003  cash            entertainment      8000.00     0.6935      欺诈⚠
INF_00004  bank_transfer   healthcare          200.00     0.7855      欺诈⚠
INF_00000  credit_card     retail               55.00     0.5920      欺诈⚠
INF_00001  wallet          travel             3200.00     0.6083      欺诈⚠


In [12]:
# ============================================================
# 总结
# ============================================================
print("=" * 60)
print("流水线总结")
print("=" * 60)

summary_items = [
    ("数据生成", f"Spark 分布式生成 {n_samples} 行"),
    ("类别特征", f"{len(cat_features)} 个: StringIndexer → OneHotEncoder"),
    ("数值特征", f"{len(num_features)} 个: Imputer(median) → StandardScaler"),
    ("训练/验证拆分", f"{train_count} / {val_count} 行"),
    ("预处理 Pipeline", f"{len(stages)} 阶段, 保存至 s3a://preprocessing/"),
    ("验证集预处理", "从 S3 加载 Pipeline → transform（复用训练参数）"),
    ("XGBoost 训练", f"2 workers, 50 trees, 耗时 {train_time_xgb:.2f}s"),
    ("模型保存", "s3a://models/xgboost_fraud_detection"),
    ("AUC (ROC)", f"{auc_roc:.4f}"),
    ("AUC (PR)", f"{auc_pr:.4f}"),
    ("Accuracy", f"{accuracy:.4f}"),
    ("F1 Score", f"{f1:.4f}"),
    ("特征维度", f"{train_feat_size}"),
]

for key, value in summary_items:
    print(f"  {key:<22}: {value}")

print("\n" + "=" * 60)
print("关键设计要点:")
print("  1. 配置文件驱动：config.yaml 定义特征列表")
print("  2. 类别/数值分离：分别用不同的预处理策略")
print("  3. Pipeline 持久化：fit 后保存到 S3")
print("  4. 验证集复用参数：从 S3 加载 PipelineModel，直接 transform")
print("  5. 全流程分布式：无 toPandas()，适合大规模数据")
print("=" * 60)

流水线总结
  数据生成                  : Spark 分布式生成 100000 行
  类别特征                  : 5 个: StringIndexer → OneHotEncoder
  数值特征                  : 20 个: Imputer(median) → StandardScaler
  训练/验证拆分               : 80060 / 19940 行
  预处理 Pipeline          : 33 阶段, 保存至 s3a://preprocessing/
  验证集预处理                : 从 S3 加载 Pipeline → transform（复用训练参数）
  XGBoost 训练            : 2 workers, 50 trees, 耗时 5.94s
  模型保存                  : s3a://models/xgboost_fraud_detection
  AUC (ROC)             : 0.4922
  AUC (PR)              : 0.9916
  Accuracy              : 0.8341
  F1 Score              : 0.9018
  特征维度                  : 48

关键设计要点:
  1. 配置文件驱动：config.yaml 定义特征列表
  2. 类别/数值分离：分别用不同的预处理策略
  3. Pipeline 持久化：fit 后保存到 S3
  4. 验证集复用参数：从 S3 加载 PipelineModel，直接 transform
  5. 全流程分布式：无 toPandas()，适合大规模数据


In [13]:
# 释放缓存
try:
    train_df.unpersist()
    val_df.unpersist()
    train_preprocessed.unpersist()
    val_preprocessed.unpersist()
    val_predictions.unpersist()
    print("所有缓存已释放")
except Exception as e:
    print(f"缓存释放: {e}")

# spark.stop()  # 取消注释以释放所有资源

所有缓存已释放
